# Few-Shot PCB Defect Segmentation Pipeline

Readable end-to-end companion for the final project. The implementation lives in `src/`; this notebook should orchestrate the official pipeline for inspection, figures, and result tables.

In [ ]:
from collections import Counter
from pathlib import Path
import csv
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from utils.config import load_yaml

config = load_yaml(PROJECT_ROOT / "configs" / "default.yaml")
config["project"]["title"]

## Dataset Readiness

Expected local artifacts are ignored by Git:

- `data/processed/VisA_pytorch/1cls` for the prepared VisA one-class layout
- `data/raw/DeepPCB-master/PCBData` for the official DeepPCB archive
- `data/manifests/*_manifest.csv` for quick train/val/test smoke runs
- `data/manifests/*_folds.csv` for the Milestone 1 five-fold CV protocol

In [ ]:
manifest_dir = PROJECT_ROOT / "data" / "manifests"
manifest_paths = {
    "visa_pcb_manifest": manifest_dir / "visa_pcb_manifest.csv",
    "deeppcb_manifest": manifest_dir / "deeppcb_manifest.csv",
    "visa_pcb_folds": manifest_dir / "visa_pcb_folds.csv",
    "deeppcb_folds": manifest_dir / "deeppcb_folds.csv",
}

for name, path in manifest_paths.items():
    if not path.exists():
        print(f"{name}: missing {path.relative_to(PROJECT_ROOT)}")
        continue
    with path.open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    split_counts = Counter(row["split"] for row in rows)
    fold_counts = Counter(row.get("fold_split", "") for row in rows if row.get("fold_split"))
    category_counts = Counter(row["category"] for row in rows)
    print(f"{name}: {len(rows)} rows")
    print("  splits:", dict(sorted(split_counts.items())))
    if fold_counts:
        print("  fold_splits:", dict(sorted(fold_counts.items())))
    print("  categories:", dict(sorted(category_counts.items())))

## Planned Execution Cells

The cells below should stay thin and call modules from `src/` as they are implemented.

### 1. Few-Shot Support Sampling

Load VisA normal training records and sample `k` support images per PCB category.

In [ ]:
# TODO: call datasets.sampling.sample_few_shot_normals after VisA manifests are generated.
k = config["few_shot"]["k"]
seed = config["few_shot"]["seed"]
k, seed

### 2. DINOv2 Feature Memory Bank

Extract patch features from the few-shot normal support set and build the normal memory bank. The runnable baseline is available as `scripts/run_dinov2_baseline.py` and can optionally add multi-scale crop fusion.

In [ ]:
# Smoke test without model weights:
# !PYTHONPATH=../src python ../scripts/run_dinov2_baseline.py \
#   --manifest ../data/manifests/visa_pcb_folds.csv \
#   --fold-id 0 --category pcb1 --k 2 --limit 2 \
#   --feature-backbone color_patch --image-size 56 --patch-size 14 \
#   --crop-sizes 128 --crop-overlap 0.25

# Real DINOv2 run after installing PyTorch in a Python 3.10-3.12 environment:
# !PYTHONPATH=../src python ../scripts/run_dinov2_baseline.py \
#   --manifest ../data/manifests/visa_pcb_folds.csv \
#   --fold-id 0 --category pcb1 --k 5 --limit 8 \
#   --feature-backbone dinov2_vits14 --crop-sizes 224,336
config["features"], config["anomaly"]

### 3. Single-Scale and Multi-Scale Anomaly Heatmaps

Run global-image inference first, then add crop-based inference and heatmap fusion for small defects. Heatmaps are saved as `.npy` files and debug overlays under `outputs/`.

In [ ]:
# The baseline command above writes scores.csv, heatmap .npy files, and overlays.
# !PYTHONPATH=../src python ../scripts/evaluate_heatmaps.py \
#   --scores-csv ../outputs/dinov2_color_patch_smoke/scores.csv \
#   --output-json ../outputs/dinov2_color_patch_smoke/metrics.json
config["multi_scale"]

### 4. SAM2 Mask Refinement

Convert anomaly maps to point/box prompts and score candidate masks. The fallback refiner is runnable now; real SAM2 requires the external SAM2 package and checkpoint.

In [ ]:
# Fallback mask-refinement smoke run:
# !PYTHONPATH=../src python ../scripts/run_mask_refinement.py \
#   --scores-csv ../outputs/dinov2_color_patch_smoke/scores.csv \
#   --output-dir ../outputs/mask_refinement_smoke \
#   --threshold 0.5 --refiner fallback
config["sam2"]

### 5. Evaluation and Figures

Report VisA segmentation metrics and DeepPCB localization/qualitative results. DeepPCB boxes are not pixel-accurate segmentation ground truth.

In [ ]:
# See metrics.json from evaluate_heatmaps.py and mask_scores.csv from run_mask_refinement.py.
config["evaluation"]["metrics"]